# 01 · ANOVA de una vía (Python)

**Semana 1 — Fundamentos y un solo factor.**

## Objetivos
- Ajustar un ANOVA de una vía para un experimento de un solo factor.
- Verificar los supuestos mediante el análisis de residuales.
- Comparar tratamientos con **Tukey HSD** y, frente a un control, con **Dunnett**.

## Paquetes
`pandas`, `numpy`, `matplotlib`, `scipy`, `statsmodels`.

> Teoría: [`../teoria/03-anova-una-via.md`](../teoria/03-anova-una-via.md) y [`../teoria/04-comparaciones-multiples.md`](../teoria/04-comparaciones-multiples.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

np.random.seed(1)  # reproducibilidad
pd.set_option('display.precision', 4)

## 1. Los datos

Resistencia a la tensión (lb/pulg²) de una fibra sintética según el **porcentaje de algodón** (factor con 5 niveles: 15, 20, 25, 30, 35 %), 5 réplicas por nivel (Montgomery, ej. 3.1).

In [ ]:
df = pd.read_csv('../datos/resistencia-algodon.csv')
df['algodon_pct'] = df['algodon_pct'].astype('category')
df.groupby('algodon_pct', observed=True)['resistencia'].describe()[['count','mean','std']]

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
df.boxplot(column='resistencia', by='algodon_pct', ax=ax, grid=False)
ax.set_title('Resistencia por % de algodón'); plt.suptitle('')
ax.set_xlabel('% algodón'); ax.set_ylabel('Resistencia'); plt.show()

La dispersión sugiere que la media crece hasta ~30 % y luego baja. Lo contrastamos formalmente con el ANOVA.

## 2. ANOVA de una vía

Modelo de efectos: $y_{ij} = \mu + \tau_i + \varepsilon_{ij}$, con $H_0: \tau_1=\cdots=\tau_5=0$.

In [ ]:
modelo = ols('resistencia ~ C(algodon_pct)', data=df).fit()
tabla = sm.stats.anova_lm(modelo, typ=2)
print(tabla)
print(f'\nR2 = {modelo.rsquared:.4f}')

Valor-p $\approx 9\times10^{-6}$: **se rechaza $H_0$**. El % de algodón afecta la resistencia y explica $R^2\approx 75\%$ de la variabilidad. Antes de comparar medias, validamos los supuestos.

## 3. Verificación de supuestos (residuales)

In [ ]:
resid = modelo.resid
ajust = modelo.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11,4))
sm.qqplot(resid, line='s', ax=ax[0])
ax[0].set_title('Q-Q normal de residuales')
ax[1].scatter(ajust, resid)
ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Valores ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()

In [ ]:
print('Shapiro-Wilk (normalidad):', stats.shapiro(resid))
grupos = [g['resistencia'].values for _, g in df.groupby('algodon_pct', observed=True)]
print('Levene (homocedasticidad):', stats.levene(*grupos))

Shapiro-Wilk ($p\approx0.18$) y Levene ($p\approx0.86$) no rechazan: no hay evidencia contra normalidad ni varianzas iguales. Los supuestos se sostienen.

## 4. Comparaciones múltiples: Tukey HSD

Tukey controla el error por familia para **todas las parejas**.

In [ ]:
tukey = pairwise_tukeyhsd(df['resistencia'], df['algodon_pct'], alpha=0.05)
print(tukey.summary())

In [ ]:
tukey.plot_simultaneous()
plt.xlabel('Resistencia media'); plt.show()

El nivel **30 %** da la mayor resistencia y supera a 15, 20 y 35 %; 15 % y 35 % no difieren entre sí.

## 5. Comparaciones contra un control: Dunnett

Tiempos de coagulación según la **dieta** (A = control; B, C, D = tratamientos). Dunnett es más potente que Tukey cuando solo interesa *tratamiento vs. control*.

In [ ]:
coag = pd.read_csv('../datos/tiempo-coagulacion.csv')
grupos_c = {k: g['tiempo'].values for k, g in coag.groupby('dieta')}
res = stats.dunnett(grupos_c['B'], grupos_c['C'], grupos_c['D'],
                    control=grupos_c['A'])
pd.DataFrame({'comparacion': ['B-A','C-A','D-A'],
              'estadistico': res.statistic,
              'p_value': res.pvalue})

Frente al control A, **B y C** difieren significativamente ($p<0.05$); **D** no.

## 6. Conclusiones

- El % de algodón **afecta** la resistencia (ANOVA, $p\approx 9\times10^{-6}$, $R^2\approx0.75$).
- Supuestos verificados (normalidad y homocedasticidad).
- **Tukey:** 30 % maximiza la resistencia.
- **Dunnett:** B y C superan al control A; D no.

> Los resultados deben coincidir con la versión en R ([`01-anova-una-via_r.ipynb`](01-anova-una-via_r.ipynb)).